# CNAME Analysis — CNAME-Based Tracker Cloaking

This notebook investigates **CNAME cloaking**, a technique where tracking domains are disguised behind first-party DNS aliases (CNAME records) to bypass ad blockers and tracker detection. For each HTTP request, the CNAME of the request hostname is resolved and re-checked against tracker lists.

## Setup and data load

In [1]:
import pandas as pd
import seaborn as sns
from utils import parse_har_files_to_dataframe
from common import BASE_HAR_DIR


sns.set_theme(style="whitegrid", context="paper")
pd.set_option('display.max_colwidth', 128)

df = parse_har_files_to_dataframe(BASE_HAR_DIR, test=False)

## Only analyze data from the tracking exp_mode

In [2]:
df = df[df.exp_mode == 'tracking']

### Detect CNAME-based evasion
- Find requests that are **not** flagged as trackers based on URL, but **are** flagged as trackers based on their CNAME
- `is_cname_tracker` is based on replacing the `req_hostname` with the CNAME and re-checking against tracker lists
- Example: `customer.etracker.com` (tracker) is the CNAME for `e.medikamente-per-klick.de` (not classified as tracker)

In [10]:
df[~df.is_tracker & df.is_cname_tracker].drop_duplicates(subset=['site_hostname', 'cname', "consent_mode"])[
    ['consent_mode', 'site_domain', 'req_hostname', 'cname',
     'url', 'is_cname_tracker', 'resource_type']]

,consent_mode,site_domain,req_hostname,cname,url,is_cname_tracker,resource_type
70492,Accept,medikamente-per-klick.de,e.medikamente-per-klick.de,customer.etracker.com,https://e.medikamente-per-klick.de/code/e.js,True,script
87783,Accept,topfarmacia.it,dmp.email.topfarmacia.it,go-direct.flx1.com,https://dmp.email.topfarmacia.it/437-23115.js?id=23115&m=437,True,script
90318,Accept,1000farmacie.it,the.sciencebehindecommerce.com,tag.device9.com,https://the.sciencebehindecommerce.com/d9core,True,script
100643,Accept,docpeter.it,dmp.mapp.docpeter.it,go-direct.flx1.com,https://dmp.mapp.docpeter.it/390-22979.js?id=22979&m=390,True,script
127657,Accept,deonlinedrogist.nl,activate.deonlinedrogist.nl,ujemkxutgo.relay.squeezely.tech,https://activate.deonlinedrogist.nl/tracker/SQ-25651742/sqzl.js,True,script
137311,Reject,medikamente-per-klick.de,e.medikamente-per-klick.de,customer.etracker.com,https://e.medikamente-per-klick.de/code/e.js,True,script
148633,Reject,topfarmacia.it,dmp.email.topfarmacia.it,go-direct.flx1.com,https://dmp.email.topfarmacia.it/437-23115.js?id=23115&m=437,True,script
159970,Reject,farmasave.it,ddbm2.paypal.com,ddbm2.paypal.com.first-party-js.datadome.co,https://ddbm2.paypal.com/tags.js,True,script
160412,Reject,docpeter.it,dmp.mapp.docpeter.it,go-direct.flx1.com,https://dmp.mapp.docpeter.it/390-22979.js?id=22979&m=390,True,script
188203,Reject,deonlinedrogist.nl,activate.deonlinedrogist.nl,ujemkxutgo.relay.squeezely.tech,https://activate.deonlinedrogist.nl/tracker/SQ-25651742/sqzl.js,True,script


### Summary: CNAME-based evasion

In [4]:
cname_evasion = df[~df.is_tracker & df.is_cname_tracker].drop_duplicates(subset=['site_hostname', 'cname'])

n_evasion_requests = df[~df.is_tracker & df.is_cname_tracker].shape[0]
n_evasion_sites = cname_evasion['site_domain'].nunique()
n_evasion_cname_domains = cname_evasion['cname_domain'].nunique()
n_evasion_pairs = cname_evasion.shape[0]

print(f"Requests evading URL-based detection via CNAME: {n_evasion_requests:,}")
print(f"Unique site–CNAME pairs:  {n_evasion_pairs}")
print(f"Sites affected:           {n_evasion_sites}")
print(f"Distinct CNAME domains:   {n_evasion_cname_domains}")
print(f"\nCNAME domains used for evasion:")
print(cname_evasion['cname_domain'].value_counts().to_string())

Requests evading URL-based detection via CNAME: 246
Unique site–CNAME pairs:  6
Sites affected:           6
Distinct CNAME domains:   5

CNAME domains used for evasion:
cname_domain
flx1.com          2
etracker.com      1
device9.com       1
squeezely.tech    1
datadome.co       1


### First-party CNAME aliases
 - Includes domains that are not labeled as trackers as well.

In [5]:
df[~df.third_party & (df.cname_domain != df.req_domain) & (df.cname != '')].\
    drop_duplicates(['site_domain', 'req_hostname', 'cname'])[
        ['consent_mode', 'site_domain', 'req_hostname', 'cname', 'is_tracker', 'is_cname_tracker']].sort_values(['is_tracker', 'is_cname_tracker'], ascending=False)

,consent_mode,site_domain,req_hostname,cname,is_tracker,is_cname_tracker
87928,Accept,topfarmacia.it,web.email.topfarmacia.it,988523091244829.wt-eu02.net,True,True
92885,Accept,efarma.com,gtm.efarma.com,ihcjdeaiapzd.eu-1.tagmanagerserverside.it,True,True
100639,Accept,docpeter.it,web.mapp.docpeter.it,184874202187158.wt-eu02.net,True,True
122017,Accept,farmaciasdirect.es,datos.farmaciasdirect.es,ghs.googlehosted.com,True,True
133597,Accept,etos.nl,v3-pixal-web.etos.nl,v3-pixal-prd.etos.nl.external.shd.eu.aholddelhaize.com,True,True
70492,Accept,medikamente-per-klick.de,e.medikamente-per-klick.de,customer.etracker.com,False,True
87783,Accept,topfarmacia.it,dmp.email.topfarmacia.it,go-direct.flx1.com,False,True
100643,Accept,docpeter.it,dmp.mapp.docpeter.it,go-direct.flx1.com,False,True
127657,Accept,deonlinedrogist.nl,activate.deonlinedrogist.nl,ujemkxutgo.relay.squeezely.tech,False,True
69493,Accept,aponeo.de,cdn-img.aponeo.de,1641998492.rsc.cdn77.org,False,False


In [6]:
df[~df.third_party & df.is_tracker & (df.cname_domain != df.req_domain) & (df.cname != '')].\
    drop_duplicates(['req_hostname', 'cname'])[['req_hostname', 'cname']]

,req_hostname,cname
70702,e.medikamente-per-klick.de,customer.etracker.com
87928,web.email.topfarmacia.it,988523091244829.wt-eu02.net
88000,dmp.email.topfarmacia.it,go-direct.flx1.com
92885,gtm.efarma.com,ihcjdeaiapzd.eu-1.tagmanagerserverside.it
94597,www.farmae.it,shops.myshopify.com
100639,web.mapp.docpeter.it,184874202187158.wt-eu02.net
100731,dmp.mapp.docpeter.it,go-direct.flx1.com
121966,www.farmaciasdirect.es,shops.myshopify.com
122017,datos.farmaciasdirect.es,ghs.googlehosted.com
133588,www.etos.nl,www.etos.nl.edgekey.net


### CNAME tracker entities
Which tracking companies are behind the CNAME aliases? We use Tracker Radar dataset for identifying entities from hostnames.

In [8]:
fp_cname_trackers = df[~df.third_party & df.is_cname_tracker & (df.cname_domain != df.req_domain) & (df.cname != '')]

entity_summary = (
    fp_cname_trackers
    .drop_duplicates(['site_domain', 'cname_domain'])
    .groupby('cname_entity')
    .agg(
        sites=('site_domain', 'nunique'),
        cname_domains=('cname_domain', 'nunique'),
    )
    .sort_values('sites', ascending=False)
)
entity_summary
# Blank cname_entities are due to unknown entities (in the Tracker Radar dataset).

,sites,cname_domains
cname_entity,,
,7,6
Webtrekk,2,1
Ahold Delhaize,1,1
Google,1,1
etracker,1,1


### CNAME-cloaked tracking by consent mode
Do CNAME-cloaked trackers persist after cookie rejection?

In [9]:
cname_by_consent = (
    fp_cname_trackers
    .drop_duplicates(['consent_mode', 'site_domain', 'cname_domain'])
    .groupby('consent_mode')
    .agg(
        sites=('site_domain', 'nunique'),
        unique_cname_domains=('cname_domain', 'nunique'),
        total_pairs=('cname_domain', 'size'),
    )
)
print("First-party CNAME-cloaked tracker presence by consent mode:\n")
cname_by_consent

First-party CNAME-cloaked tracker presence by consent mode:



,sites,unique_cname_domains,total_pairs
consent_mode,,,
Accept,8,10,13
Reject,8,10,13
